In [ ]:
import torch.nn as nn

### Defining the Model

For this section, we are defining the model being used which is a simple one layer transformer model. The main thing that differentiates this from other transformers is in the self attention mechanism where I did not use masking to ensure causal attention which may be more useful in data that comprises of long sequences of words. I was curious to see if this would help this model learn more efficiently here. However, I did use masking to prevent token embeddings from attending to padding emeddeings which I added to ensure that all words were the same length.

In [ ]:
class POS_Transformer (nn.Module):
    def __init__ (self):
        super().__init__()
        self.MHA = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.MLP = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim)
        )
        self.layernorm1 = nn.LayerNorm(embed_dim)
        self.layernorm2 = nn.LayerNorm(embed_dim)
        self.unembed = nn.Linear(embed_dim, num_pos_tags)
        
    def forward (self, x, attention_mask):
        batch_size = x.shape[0]
        seq_len = x.shape[1]
        embed_dim = x.shape[2]

        attention_out = self.MHA(x, x, x, key_padding_mask=attention_mask)[0]
        x = self.layernorm1(attention_out + x)
        mlp_out = self.MLP(x)
        x = self.layernorm2(mlp_out + x)
        return self.unembed(x)
    
    def backward (self, grad):
        return grad